# Deep Learning with PyTorch — Course Summary

**IBM AI Engineering Professional Certificate — Course 4**

This notebook provides a **detailed summary** of all lab materials from the *Deep Learning with PyTorch* course. 

---
## Summary Table: Topics and Key Notebooks
---

| Topic | Notebooks | Main idea |
|-------|-----------|-----------|
| Logistic regression | bad_inshilization_logistic_regression_with_mean_square_error_v2, cross_entropy_logistic_regression_v2 | Use cross-entropy, not MSE; good initialization matters |
| Softmax | softmax_in_one_dimension_v2, lab_predicting_mnist_using_softmax_v2 | Multi-class logits + CrossEntropyLoss; baseline on MNIST |
| Shallow NNs | simple1hiddenlayer, multiple_neurons, xor_v2, one_layer_neural_network_mnist | One hidden layer, non-linear decision boundaries, MNIST |
| Activations | activationfuction_v2, mist1layer_v2 | Sigmoid, Tanh, ReLU; ReLU preferred for hidden layers |
| Deep NNs | mist2layer_v2, mulitclassspiralrulu_v2 | Two+ layers; nn.ModuleList for flexible depth |
| Initialization | initializationsame, xaviermist1layer_v2, he_initialization_v2 | Avoid same weights; Xavier for tanh, He for ReLU |
| Momentum | momentumwithpolynomialfunctions_v2, neuralnetworkswithmomentum_v2 | SGD with momentum for faster, stabler training |
| BatchNorm | bachnorm_v2 | Normalize activations; faster convergence, stability |
| Convolution basics | what_is_convolution, activation_max_pooling_1, multiple_channel_convolution | Conv, stride, padding; activation; multi-channel; max pool |
| CNN practice | convolutionalneralnetworksimple_example, cnn_small_image, cnn_small_image_batch, convolutional_neural_network_for_anime_image_classification | Small CNN, MNIST, BatchNorm in CNN, custom image dataset |

## Lab-by-Lab Summary (Objective, Strategy, Consolidated Code)
---

For each lab (excluding the Final Fashion MNIST Project), the following format is used:
- **Objective:** Extracted from the lab's Objective section.
- **Implementation Strategy:** How the code achieves the objective (model, optimizer, data).
- **Consolidated Code Block:** Single runnable cell with imports, dataset, model, and training loop (no extra prints/plots).

<a id='bad_inshilization_logistic_regression_with_mean_square_error_v2'></a>

### Lab: Bad Initialization & MSE (bad_inshilization_logistic_regression_with_mean_square_error_v2)

**Objective:** Understand how bad initialization and using Mean Square Error (MSE) as the loss for logistic regression can hurt model accuracy.

**Implementation Strategy:**
- Uses a simple synthetic 1D dataset and a custom `Dataset` with `DataLoader` for batching.
- Defines logistic regression as a single `nn.Linear` layer with sigmoid; loss is **MSE** (not cross-entropy).
- Trains with **SGD**; with poor initial weights, the model can get stuck due to vanishing gradients in flat regions of the MSE loss.
- Demonstrates that the same setup may fail to converge, motivating the switch to cross-entropy in the next lab.

In [14]:
# Consolidated: Logistic regression with MSE + bad init (bad_inshilization_logistic_regression_with_mean_square_error_v2)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

class Data(Dataset):
    def __init__(self):
        self.x = torch.arange(-1, 1, 0.1).view(-1, 1)
        self.y = torch.zeros(self.x.shape[0], 1)
        self.y[self.x[:, 0] > 0.2] = 1
        self.len = self.x.shape[0]
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    def __len__(self):
        return self.len

class LogisticRegression(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()
        self.linear = nn.Linear(n_inputs, 1)
    def forward(self, x):
        return torch.sigmoid(self.linear(x))

data_set = Data()
model = LogisticRegression(1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(dataset=data_set, batch_size=3)

for epoch in range(100):
    for x, y in loader:
        yhat = model(x)
        loss = criterion(yhat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

acc = (model(data_set.x) > 0.5).float().eq(data_set.y).float().mean().item()

<a id='cross_entropy_logistic_regression_v2'></a>

### Lab: Cross-Entropy Logistic Regression (cross_entropy_logistic_regression_v2)

**Objective:** See how Cross-Entropy with random initialization influences the accuracy of the model (compared to MSE).

**Implementation Strategy:**
- Same synthetic 1D dataset and custom `Dataset`/`DataLoader` as in bad_inshilization_logistic_regression_with_mean_square_error_v2.
- Logistic regression model: one `nn.Linear` with sigmoid; loss is **binary cross-entropy** (e.g. custom or `nn.BCELoss()`).
- Trains with **SGD**; cross-entropy gives a gradient that does not vanish when the sigmoid saturates, so training converges well even with suboptimal init.
- Single runnable cell below uses a manual BCE-style criterion for clarity.

In [15]:
# Consolidated: Logistic regression with cross-entropy (cross_entropy_logistic_regression_v2)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

class Data(Dataset):
    def __init__(self):
        self.x = torch.arange(-1, 1, 0.1).view(-1, 1)
        self.y = torch.zeros(self.x.shape[0], 1)
        self.y[self.x[:, 0] > 0.2] = 1
        self.len = self.x.shape[0]
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    def __len__(self):
        return self.len

class LogisticRegression(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()
        self.linear = nn.Linear(n_inputs, 1)
    def forward(self, x):
        return torch.sigmoid(self.linear(x))

def bce_criterion(yhat, y):
    return -torch.mean(y * torch.log(yhat + 1e-8) + (1 - y) * torch.log(1 - yhat + 1e-8))

data_set = Data()
model = LogisticRegression(1)
optimizer = torch.optim.SGD(model.parameters(), lr=2)
loader = DataLoader(dataset=data_set, batch_size=3)

for epoch in range(100):
    for x, y in loader:
        yhat = model(x)
        loss = bce_criterion(yhat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

acc = (model(data_set.x) > 0.5).float().eq(data_set.y).float().mean().item()

<a id='softmax_in_one_dimension_v2'></a>

### Lab: Softmax Classifier 1D (softmax_in_one_dimension_v2)

**Objective:** Build a Softmax classifier using the Sequential module in PyTorch for three linearly separable classes in one dimension.

**Implementation Strategy:**
- Custom `Dataset` with 1D inputs and three class labels (0, 1, 2); data created with `torch.arange` and thresholding.
- Model: `nn.Sequential(nn.Linear(1, 3))` — linear layer outputs 3 logits; CrossEntropyLoss applies log-softmax internally.
- **SGD** optimizer and **DataLoader** for minibatch training.
- Training loop: forward, `CrossEntropyLoss`, backward, step; no softmax needed in the model (handled by the loss).

In [16]:
# Consolidated: Softmax classifier 1D (softmax_in_one_dimension_v2)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

class Data(Dataset):
    def __init__(self):
        self.x = torch.arange(-2, 2, 0.1).view(-1, 1)
        self.y = torch.zeros(self.x.shape[0], dtype=torch.long)
        self.y[(self.x > -1.0)[:, 0] & (self.x < 1.0)[:, 0]] = 1
        self.y[(self.x >= 1.0)[:, 0]] = 2
        self.len = self.x.shape[0]
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    def __len__(self):
        return self.len

model = nn.Sequential(nn.Linear(1, 3))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
data_set = Data()
loader = DataLoader(dataset=data_set, batch_size=5)

for epoch in range(200):
    for x, y in loader:
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

pred = model(data_set.x).argmax(dim=1)
acc = (pred == data_set.y).float().mean().item()

<a id='lab_predicting_mnist_using_softmax_v2'></a>

### Lab: MNIST with Softmax (lab_predicting_mnist_using_softmax_v2)

**Objective:** Classify handwritten digits from the MNIST database using a Softmax classifier (single linear layer).

**Implementation Strategy:**
- **Dataset:** `torchvision.datasets.MNIST` with `transform=transforms.ToTensor()`; images flattened or kept as 28×28 and flattened in the model.
- **Model:** Single `nn.Linear(784, 10)` (or equivalent) producing 10 logits; `nn.CrossEntropyLoss()` for training.
- **Optimizer:** SGD (or Adam); **DataLoader** with batch_size (e.g. 100) for train and validation.
- Training loop: iterate over train_loader, compute loss, zero_grad, backward, step; optionally validate with model.eval() and torch.no_grad().

In [18]:
# Consolidated: MNIST Softmax classifier (lab_predicting_mnist_using_softmax_v2)
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
torch.manual_seed(0)

train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
validation_dataset = dsets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=100, shuffle=True)
validation_loader = torch.utils.data.DataLoader(dataset=validation_dataset, batch_size=5000)

class SoftMax(nn.Module):
    def __init__(self, in_size, out_size):
        super().__init__()
        self.linear = nn.Linear(in_size, out_size)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.linear(x)

model = SoftMax(784, 10)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(10):
    for x, y in train_loader:
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    correct = 0
    for x, y in validation_loader:
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
acc = correct / len(validation_dataset)

100.0%
100.0%
100.0%
100.0%


<a id='simple1hiddenlayer'></a>

### Lab: Simple One-Hidden-Layer Network (simple1hiddenlayer)

**Objective:** Create a simple neural network in PyTorch with one hidden layer for non-linearly separable data.

**Implementation Strategy:**
- Synthetic 1D data and custom `Dataset`/`DataLoader`; target is binary or multi-class.
- **Model:** `nn.Sequential` or custom `nn.Module` with one hidden layer (e.g. Linear → activation) and output layer; activation (e.g. sigmoid or tanh) applied after the hidden layer.
- **Loss:** CrossEntropyLoss (multi-class) or BCELoss (binary); **SGD** optimizer.
- Standard training loop: zero_grad, forward, loss, backward, step.

In [ ]:
# Consolidated: Simple one-hidden-layer net (simple1hiddenlayer)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

class Data(Dataset):
    def __init__(self):
        self.x = torch.arange(-1, 1, 0.1).view(-1, 1)
        self.y = (self.x[:, 0] > 0).float().unsqueeze(1)
        self.len = self.x.shape[0]
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    def __len__(self):
        return self.len

model = nn.Sequential(
    nn.Linear(1, 4),
    nn.Sigmoid(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(dataset=Data(), batch_size=5)

for epoch in range(200):
    for x, y in loader:
        yhat = model(x)
        loss = criterion(yhat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='multiple_neurons'></a>

### Lab: Multiple Neurons (multiple_neurons)

**Objective:** Create a more complex neural network in PyTorch with multiple hidden neurons to fit richer patterns.

**Implementation Strategy:**
- Same pattern as simple1hiddenlayer: custom or synthetic data, `Dataset`, `DataLoader`.
- **Model:** One hidden layer with **more units** (e.g. 8 or 16) and an activation (sigmoid/tanh); output layer for classification or regression.
- **Loss** and **SGD** as before; training loop unchanged.
- More neurons allow the network to approximate more complex decision boundaries.

In [ ]:
# Consolidated: Multiple neurons in hidden layer (multiple_neurons)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

class Data(Dataset):
    def __init__(self):
        self.x = torch.arange(-1, 1, 0.05).view(-1, 1)
        self.y = (torch.sin(self.x[:, 0] * 3) > 0).float().unsqueeze(1)
        self.len = self.x.shape[0]
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    def __len__(self):
        return self.len

model = nn.Sequential(
    nn.Linear(1, 16),
    nn.Tanh(),
    nn.Linear(16, 1),
    nn.Sigmoid()
)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(dataset=Data(), batch_size=10)

for epoch in range(300):
    for x, y in loader:
        yhat = model(x)
        loss = criterion(yhat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='xor_v2'></a>

### Lab: XOR with Neural Network (xor_v2)

**Objective:** Create a neural network model with multiple neurons to classify the XOR problem (non-linearly separable).

**Implementation Strategy:**
- **Data:** Four points for XOR: (0,0)→0, (0,1)→1, (1,0)→1, (1,1)→0; custom `Dataset` and `DataLoader`.
- **Model:** One hidden layer (at least 2 neurons) with sigmoid/tanh and binary output with sigmoid; or two-class logits with CrossEntropyLoss.
- **SGD** and standard training loop; demonstrates that a single hidden layer can learn XOR.

In [ ]:
# Consolidated: XOR with one-hidden-layer net (xor_v2)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

class Data(Dataset):
    def __init__(self):
        self.x = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
        self.y = torch.tensor([0., 1., 1., 0.]).unsqueeze(1)
        self.len = 4
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    def __len__(self):
        return self.len

model = nn.Sequential(
    nn.Linear(2, 4),
    nn.Sigmoid(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)
loader = DataLoader(dataset=Data(), batch_size=4)

for epoch in range(2000):
    for x, y in loader:
        yhat = model(x)
        loss = criterion(yhat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='one_layer_neural_network_mnist'></a>

### Lab: One-Layer Neural Network on MNIST (one_layer_neural_network_mnist)

**Objective:** Classify handwritten digits using a neural network with one hidden layer (no convolution).

**Implementation Strategy:**
- **Dataset:** MNIST via `torchvision.datasets.MNIST` with `ToTensor()`; images flattened to 784-D.
- **Model:** `nn.Linear(784, hidden_size)` → activation (e.g. ReLU or Tanh) → `nn.Linear(hidden_size, 10)`; **CrossEntropyLoss**.
- **DataLoader** with batch_size and shuffle; **SGD** (or Adam) optimizer.
- Training loop over train_loader; validation with model.eval() and no_grad().

In [ ]:
# Consolidated: One-hidden-layer NN on MNIST (one_layer_neural_network_mnist)
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
torch.manual_seed(0)

train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
validation_dataset = dsets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=2000, shuffle=True)
validation_loader = torch.utils.data.DataLoader(dataset=validation_dataset, batch_size=5000)

class Net(nn.Module):
    def __init__(self, D_in, H, D_out):
        super().__init__()
        self.linear1 = nn.Linear(D_in, H)
        self.linear2 = nn.Linear(H, D_out)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.sigmoid(self.linear1(x))
        return self.linear2(x)

model = Net(784, 100, 10)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(10):
    for x, y in train_loader:
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    correct = sum((model(x).argmax(dim=1) == y).sum().item() for x, y in validation_loader)
acc = correct / len(validation_dataset)

<a id='activationfuction_v2'></a>

### Lab: Activation Functions (activationfuction_v2)

**Objective:** Apply different activation functions (Sigmoid, Tanh, ReLU) in a neural network and compare their effect.

**Implementation Strategy:**
- Small synthetic or toy data; custom `Dataset`/`DataLoader`.
- **Model:** Same architecture (e.g. one or two hidden layers) but with different activations (sigmoid, tanh, relu) in separate model definitions or a parameterized forward.
- **Loss** (MSE or CrossEntropy) and **SGD**; training loop identical; comparison is done by training each variant and observing convergence or final accuracy.

In [ ]:
# Consolidated: Activation functions demo (activationfuction_v2)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

class Data(Dataset):
    def __init__(self):
        self.x = torch.randn(100, 2)
        self.y = (self.x.sum(dim=1) > 0).long()
        self.len = 100
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

def make_net(activation):
    act = nn.Sigmoid if activation == 'sigmoid' else (nn.Tanh if activation == 'tanh' else nn.ReLU)
    return nn.Sequential(nn.Linear(2, 8), act(), nn.Linear(8, 2))

model = make_net('relu')
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(Data(), batch_size=10)
for epoch in range(100):
    for x, y in loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='mist1layer_v2'></a>

### Lab: MNIST One Layer with Activations (mist1layer_v2)

**Objective:** Apply different activation functions (Sigmoid, Tanh, ReLU) on the MNIST dataset with a one-hidden-layer network and compare results.

**Implementation Strategy:**
- **Dataset:** MNIST with ToTensor(); DataLoader with batch_size and shuffle.
- **Model:** Linear(784, H) → activation (Sigmoid / Tanh / ReLU) → Linear(H, 10); CrossEntropyLoss.
- **SGD** optimizer; training loop and validation as in one_layer_neural_network_mnist; lab typically compares all three activations (separate runs or separate models).

In [ ]:
# Consolidated: MNIST one layer with ReLU (mist1layer_v2)
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
torch.manual_seed(0)

train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=2000, shuffle=True)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(784, 100)
        self.linear2 = nn.Linear(100, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.linear1(x))
        return self.linear2(x)

model = Net()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
for epoch in range(10):
    for x, y in train_loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='mist2layer_v2'></a>

### Lab: MNIST Two-Layer Network (mist2layer_v2)

**Objective:** Define several neural networks (Sigmoid, Tanh, ReLU), criterion, optimizer; test them on MNIST with two hidden layers and analyze results.

**Implementation Strategy:**
- **Dataset:** MNIST with ToTensor(); DataLoader for train and validation.
- **Model:** Two hidden layers (e.g. 784→H1→H2→10) with activation (Sigmoid, Tanh, or ReLU) after each hidden layer; CrossEntropyLoss.
- **SGD** optimizer; standard training loop; validation with model.eval() and no_grad(); lab compares the three activation variants.

In [ ]:
# Consolidated: MNIST two-layer ReLU (mist2layer_v2)
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
torch.manual_seed(0)

train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=2000, shuffle=True)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(784, 100)
        self.linear2 = nn.Linear(100, 50)
        self.linear3 = nn.Linear(50, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.linear1(x))
        x = torch.relu(self.linear2(x))
        return self.linear3(x)

model = Net()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
for epoch in range(10):
    for x, y in train_loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='mulitclassspiralrulu_v2'></a>

### Lab: Multiclass Spiral with ReLU (mulitclassspiralrulu_v2)

**Objective:** Build a deeper network (e.g. with `nn.ModuleList`) for multiclass spiral data and train with ReLU.

**Implementation Strategy:**
- **Data:** Spiral or similar multi-class 2D data; custom `Dataset`/`DataLoader`.
- **Model:** Multiple hidden layers (e.g. using ModuleList or Sequential) with ReLU; output layer has num_classes logits; CrossEntropyLoss.
- **SGD** or Adam; standard training loop; demonstrates that depth + ReLU can learn complex boundaries.

In [ ]:
# Consolidated: Multiclass spiral with ReLU (mulitclassspiralrulu_v2)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

class SpiralData(Dataset):
    def __init__(self, n=500):
        self.x = torch.randn(n, 2) * 2
        r = self.x.norm(dim=1)
        self.y = ((torch.atan2(self.x[:, 1], self.x[:, 0]) + 3.14159) / (2 * 3.14159) * 3).long().clamp(0, 2)
        self.len = n
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

model = nn.Sequential(
    nn.Linear(2, 32), nn.ReLU(),
    nn.Linear(32, 32), nn.ReLU(),
    nn.Linear(32, 3)
)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loader = DataLoader(SpiralData(), batch_size=20)
for epoch in range(100):
    for x, y in loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='initializationsame'></a>

### Lab: Same Initialization (initializationsame)

**Objective:** Show that initializing all weights to the same value is harmful (e.g. symmetric breaking problem).

**Implementation Strategy:**
- Small network (e.g. one hidden layer) on toy or MNIST data; custom init that sets all weights to the same constant.
- **Model:** Same architecture as a baseline but with identical initial weights per layer; **SGD** and CrossEntropyLoss.
- Training loop standard; comparison with proper random init shows that same-value init leads to poor or no learning.

In [ ]:
# Consolidated: Same init (bad) vs default init (initializationsame)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as dsets
torch.manual_seed(0)

train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_dataset, batch_size=2000, shuffle=True)

class Net(nn.Module):
    def __init__(self, same_init=False):
        super().__init__()
        self.l1 = nn.Linear(784, 50)
        self.l2 = nn.Linear(50, 10)
        if same_init:
            nn.init.constant_(self.l1.weight, 0.01)
            nn.init.constant_(self.l1.bias, 0.)
            nn.init.constant_(self.l2.weight, 0.01)
            nn.init.constant_(self.l2.bias, 0.)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.l1(x))
        return self.l2(x)

model = Net(same_init=False)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
for epoch in range(5):
    for x, y in train_loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='xaviermist1layer_v2'></a>

### Lab: Xavier Initialization on MNIST (xaviermist1layer_v2)

**Objective:** Compare Uniform, default, and Xavier initialization on MNIST with a one-layer (one hidden layer) network and tanh.

**Implementation Strategy:**
- **Dataset:** MNIST; DataLoader as in previous MNIST labs.
- **Model:** One hidden layer with **tanh**; output 10 classes; one model uses **Xavier** init (`nn.init.xavier_uniform_` or similar) for linear layers.
- **SGD**, CrossEntropyLoss; training loop; comparison shows Xavier often improves convergence for tanh.

In [ ]:
# Consolidated: Xavier init on MNIST 1-layer tanh (xaviermist1layer_v2)
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
torch.manual_seed(0)

train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=2000, shuffle=True)

class Net_Xavier(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(784, 100)
        self.linear2 = nn.Linear(100, 10)
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.tanh(self.linear1(x))
        return self.linear2(x)

model = Net_Xavier()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
for epoch in range(10):
    for x, y in train_loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='he_initialization_v2'></a>

### Lab: He Initialization (he_initialization_v2)

**Objective:** Compare Uniform, default, and He initialization on MNIST with ReLU (one hidden layer).

**Implementation Strategy:**
- **Dataset:** MNIST; DataLoader as before.
- **Model:** One hidden layer with **ReLU**; **He** init (`nn.init.kaiming_uniform_`) applied to linear layers to suit ReLU.
- **SGD**, CrossEntropyLoss; training loop; He initialization helps avoid vanishing/exploding activations with ReLU.

In [ ]:
# Consolidated: He init on MNIST 1-layer ReLU (he_initialization_v2)
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
torch.manual_seed(0)

train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=2000, shuffle=True)

class Net_He(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(784, 100)
        self.linear2 = nn.Linear(100, 10)
        nn.init.kaiming_uniform_(self.linear1.weight, nonlinearity='relu')
        nn.init.kaiming_uniform_(self.linear2.weight, nonlinearity='relu')
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.linear1(x))
        return self.linear2(x)

model = Net_He()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
for epoch in range(10):
    for x, y in train_loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='momentumwithpolynomialfunctions_v2'></a>

### Lab: Momentum with Polynomial Functions (momentumwithpolynomialfunctions_v2)

**Objective:** Understand saddle points, local minima, and noisy gradients; see how momentum helps optimization on a simple polynomial objective.

**Implementation Strategy:**
- **No dataset:** A simple scalar or low-dimensional objective (e.g. polynomial) is defined; parameters are learned by minimizing this objective.
- **Model:** Often a single parameter or small MLP whose output is fed into a loss (e.g. (pred - target)^2); or parameters updated directly on the polynomial loss.
- **SGD with momentum** (e.g. `torch.optim.SGD([param], lr=..., momentum=0.9)`) vs **SGD without momentum**; same training loop (forward, loss, backward, step); momentum helps escape flat regions and dampen oscillations.

In [ ]:
# Consolidated: Momentum on polynomial objective (momentumwithpolynomialfunctions_v2)
import torch
import torch.nn as nn
torch.manual_seed(0)

class OneParam(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.tensor([2.0]))
    def forward(self, x):
        return self.w * x

x = torch.tensor([1.0])
model = OneParam()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
for step in range(100):
    y = model(x)
    loss = (y - 5.0) ** 2
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

<a id='neuralnetworkswithmomentum_v2'></a>

### Lab: Neural Networks with Momentum (neuralnetworkswithmomentum_v2)

**Objective:** Train neural networks with SGD using different momentum values and compare convergence (e.g. on spiral or MNIST).

**Implementation Strategy:**
- **Data:** Spiral or MNIST; custom Dataset/DataLoader or torchvision.
- **Model:** One or two hidden layers with ReLU (or tanh); CrossEntropyLoss.
- **Optimizer:** `torch.optim.SGD(..., momentum=0.0)` vs `momentum=0.5` vs `momentum=0.9`; same training loop; compare loss curves or final accuracy to show benefit of momentum.

In [ ]:
# Consolidated: NN with momentum (neuralnetworkswithmomentum_v2)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

class Data(Dataset):
    def __init__(self):
        self.x = torch.randn(200, 2) * 1.5
        self.y = (self.x[:, 0] * self.x[:, 1] > 0).long()
        self.len = len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

model = nn.Sequential(
    nn.Linear(2, 20), nn.ReLU(),
    nn.Linear(20, 2)
)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
loader = DataLoader(Data(), batch_size=20)
for epoch in range(50):
    for x, y in loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='bachnorm_v2'></a>

### Lab: Batch Normalization (bachnorm_v2)

**Objective:** Compare a network with Batch Normalization to a regular network on MNIST (e.g. two hidden layers).

**Implementation Strategy:**
- **Dataset:** MNIST; DataLoader as in previous MNIST labs.
- **Model (no BN):** Linear → ReLU → Linear → ReLU → Linear; **Model (with BN):** Linear → **BatchNorm1d** → ReLU → Linear → **BatchNorm1d** → ReLU → Linear; CrossEntropyLoss.
- **SGD** (or Adam); standard training loop; use model.train() during training and model.eval() for validation so BatchNorm uses running stats when evaluating.

In [ ]:
# Consolidated: MNIST with BatchNorm (bachnorm_v2)
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
torch.manual_seed(0)

train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=2000, shuffle=True)

class NetBatchNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(784, 100)
        self.bn1 = nn.BatchNorm1d(100)
        self.linear2 = nn.Linear(100, 50)
        self.bn2 = nn.BatchNorm1d(50)
        self.linear3 = nn.Linear(50, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.bn1(self.linear1(x)))
        x = torch.relu(self.bn2(self.linear2(x)))
        return self.linear3(x)

model = NetBatchNorm()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
for epoch in range(10):
    model.train()
    for x, y in train_loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='what_is_convolution'></a>

### Lab: What is Convolution (what_is_convolution)

**Objective:** Learn about convolution, how to determine the size of the output, and the effect of stride and zero padding.

**Implementation Strategy:**
- **No training:** Uses `nn.Conv2d` (and possibly `F.conv2d`) with sample tensors to demonstrate output shape.
- **Formula:** Output size depends on input size, kernel size, stride, and padding; often demonstrated with a single conv layer and `.shape` or a helper that computes $(W - K + 2P)/S + 1$.
- Consolidated code: imports, create a small 4D tensor (N,C,H,W), apply Conv2d, print input and output shapes (and optionally stride/padding effect).

In [ ]:
# Consolidated: Convolution output size (what_is_convolution)
import torch
import torch.nn as nn
torch.manual_seed(0)

conv = nn.Conv2d(in_channels=1, out_channels=2, kernel_size=3, stride=1, padding=0)
x = torch.randn(1, 1, 8, 8)
y = conv(x)
# Output size: (8 - 3) / 1 + 1 = 6
assert y.shape == (1, 2, 6, 6)

conv_pad = nn.Conv2d(1, 2, kernel_size=3, stride=1, padding=1)
y_pad = conv_pad(x)
# With padding=1: (8 + 2 - 3) / 1 + 1 = 8
assert y_pad.shape == (1, 2, 8, 8)

<a id='activation_max_pooling_1'></a>

### Lab: Activation and Max Pooling (activation_max_pooling_1)

**Objective:** Learn how to apply an activation function after convolution and how to use max pooling.

**Implementation Strategy:**
- **No full training:** Demonstrates Conv2d → activation (ReLU) → MaxPool2d on a sample tensor; shows how pooling reduces spatial dimensions.
- **Layers:** `nn.Conv2d`, `nn.ReLU()` (or F.relu), `nn.MaxPool2d(kernel_size=2)` (and optionally output shape).
- Consolidated code: build a small Sequential(Conv2d, ReLU, MaxPool2d), run one forward pass, print shapes.

In [ ]:
# Consolidated: Conv + ReLU + MaxPool (activation_max_pooling_1)
import torch
import torch.nn as nn
torch.manual_seed(0)

layer = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2)
)
x = torch.randn(2, 1, 8, 8)
y = layer(x)
# After conv+pad: 8x8; after pool: 4x4
assert y.shape == (2, 4, 4, 4)

<a id='multiple_channel_convolution'></a>

### Lab: Multiple Input and Output Channels (multiple_channel_convolution)

**Objective:** Learn multiple input and multiple output channels in convolution.

**Implementation Strategy:**
- **No full training:** Uses Conv2d with `in_channels>1` and `out_channels>1` on a tensor of shape (N, C_in, H, W); shows output (N, C_out, H', W').
- Explains how each output channel is a sum of convolutions over all input channels; consolidated code: one Conv2d with multiple in/out channels, one forward pass, print shapes.

In [ ]:
# Consolidated: Multiple channels (multiple_channel_convolution)
import torch
import torch.nn as nn
torch.manual_seed(0)

conv = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, padding=1)
x = torch.randn(2, 3, 10, 10)
y = conv(x)
assert y.shape == (2, 8, 10, 10)

<a id='convolutionalneralnetworksimple_example'></a>

### Lab: CNN Simple Example (convolutionalneralnetworksimple_example)

**Objective:** Learn Convolutional Neural Networks; define Softmax, criterion, optimizer, and train the model (e.g. on a toy task like horizontal vs vertical lines).

**Implementation Strategy:**
- **Data:** Small images (e.g. 11×11) with horizontal or vertical lines; custom Dataset or synthetic tensors; DataLoader.
- **Model:** Small CNN (e.g. Conv2d → ReLU → pool → flatten → Linear → 2 or 10 classes); CrossEntropyLoss.
- **SGD** (or Adam); standard training loop; demonstrates full pipeline: load data, CNN forward, loss, backward, step.

In [ ]:
# Consolidated: CNN simple toy (convolutionalneralnetworksimple_example)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

class ToyImageData(Dataset):
    def __init__(self, n=100):
        self.x = torch.randn(n, 1, 11, 11)
        self.x[:, :, 5, :] += 1
        self.y = (self.x[:, 0].mean(dim=1) > 0).long()
        self.len = n
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 4, 3)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(4 * 4 * 4, 2)
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loader = DataLoader(ToyImageData(), batch_size=10)
for epoch in range(20):
    for x, y in loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='cnn_small_image'></a>

### Lab: CNN on Small Images / MNIST (cnn_small_image)

**Objective:** Use a Convolutional Neural Network to classify MNIST digits and learn how to reshape images to process faster (e.g. resize to 16×16).

**Implementation Strategy:**
- **Dataset:** MNIST with ToTensor(); optionally `transforms.Resize((16,16))` to reduce size; DataLoader.
- **Model:** CNN with Conv2d layers, ReLU, MaxPool2d, then flatten and Linear to 10 classes; CrossEntropyLoss.
- **SGD** or Adam; full training loop; validation with model.eval() and no_grad().

In [ ]:
# Consolidated: CNN on MNIST (cnn_small_image)
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
torch.manual_seed(0)

train_dataset = dsets.MNIST(root='./data', train=True, download=True,
    transform=transforms.Compose([transforms.Resize((16, 16)), transforms.ToTensor()]))
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=100, shuffle=True)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.fc = nn.Linear(16 * 4 * 4, 10)
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
for epoch in range(5):
    for x, y in train_loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='cnn_small_image_batch'></a>

### Lab: CNN with Batch Normalization (cnn_small_image_batch)

**Objective:** Compare a CNN using Batch Normalization with a regular CNN to classify MNIST digits.

**Implementation Strategy:**
- **Dataset:** MNIST (optionally resized); DataLoader as in cnn_small_image.
- **Model (with BN):** Conv2d → **BatchNorm2d** → ReLU → Pool (and possibly more conv+bn+relu+pool blocks) → flatten → Linear; CrossEntropyLoss.
- **SGD** or Adam; training loop with model.train()/model.eval() so BatchNorm uses batch stats vs running stats correctly.

In [ ]:
# Consolidated: CNN with BatchNorm on MNIST (cnn_small_image_batch)
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
torch.manual_seed(0)

train_dataset = dsets.MNIST(root='./data', train=True, download=True,
    transform=transforms.Compose([transforms.Resize((16, 16)), transforms.ToTensor()]))
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=100, shuffle=True)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(8)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.fc = nn.Linear(16 * 4 * 4, 10)
    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
for epoch in range(5):
    model.train()
    for x, y in train_loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

<a id='convolutional_neural_network_for_anime_image_classification'></a>

### Lab: Convolutional Neural Network for Anime Image Classification (convolutional_neural_network_for_anime_image_classification)

**Objective:** Build and train a CNN for image classification on a custom dataset (e.g. anime character or class images).

**Implementation Strategy:**
- **Dataset:** Custom `Dataset` that loads images from folders (e.g. by class); transforms (Resize, ToTensor, Normalize); DataLoader with batch_size and shuffle.
- **Model:** CNN (several Conv2d + BatchNorm + ReLU + Pool, then flatten and Linear layers) outputting num_classes logits; CrossEntropyLoss.
- **Optimizer:** SGD or Adam; full training loop; validation with model.eval() and torch.no_grad(); optional checkpointing.

In [ ]:
# Consolidated: CNN for custom image classification (convolutional_neural_network_for_anime_image_classification)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import torchvision.transforms as T
torch.manual_seed(0)

# Minimal example: random tensors as placeholder when no image dir is available
class DummyImageDataset(Dataset):
    def __init__(self, n=200, num_classes=5, size=64):
        self.x = torch.randn(n, 3, size, size)
        self.y = torch.randint(0, num_classes, (n,))
        self.len = n
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

class CNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(64, num_classes)
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

train_loader = DataLoader(DummyImageDataset(n=200, num_classes=5, size=64), batch_size=16, shuffle=True)
model = CNN(num_classes=5)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(5):
    for x, y in train_loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

---
## PyTorch Workflow (Recap) — Functions, Usage, and Parameters
---

### 1. Data

**`torch.utils.data.Dataset`** (abstract base class)  
- **What it is:** Interface for a collection of samples (inputs, labels). You subclass it and implement `__len__` and `__getitem__(self, index)`.  
- **How used:** Store your data; `DataLoader` calls `__getitem__` to fetch batches.  
- **Parameters / methods:**  
  - `__len__(self)` → return number of samples.  
  - `__getitem__(self, index)` → return the `index`-th sample (e.g. `(x, y)` tensor pair).

**`torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, ...)`**  
- **What it is:** Iterates over `dataset` in minibatches.  
- **How used:** `for x, y in train_loader:` to get batches for training.  
- **Key parameters:**  
  - `dataset`: your `Dataset` instance.  
  - `batch_size`: number of samples per batch.  
  - `shuffle`: if `True`, shuffle indices each epoch (use `True` for training).  
  - `num_workers`: number of subprocesses for loading data (0 = main process).

**Transforms (e.g. `torchvision.transforms`)**  
- **`transforms.ToTensor()`:** converts PIL/numpy array to `torch.Tensor` and scales to $[0,1]$ if needed. No constructor parameters.  
- **`transforms.Resize(size)`:** resizes image. `size` can be int (shorter side) or `(H, W)`.  
- **How used:** Pass as `transform=...` when building the dataset so each `__getitem__` applies the transform.

---

### 2. Model

**`nn.Module`** (base class for all layers and models)  
- **What it is:** Base class for any learnable component. You subclass and define layers in `__init__` and the forward pass in `forward()`.  
- **How used:** Define your network; call `model(x)` to run a forward pass (PyTorch calls `forward` automatically).  
- **Parameters / methods:**  
  - `__init__(self)`: register layers with `self.linear = nn.Linear(...)` so parameters are tracked.  
  - `forward(self, x)`: return output tensor(s).  
  - `.parameters()`: generator over all learnable parameters (used by the optimizer).

**`nn.Linear(in_features, out_features, bias=True)`**  
- **What it is:** Fully connected layer $y = xW^T + b$.  
- **Parameters:** `in_features`, `out_features` (input/output sizes), `bias` (default `True`).

**`nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, ...)`**  
- **Parameters:** `in_channels`, `out_channels`, `kernel_size` (int or tuple), `stride`, `padding`.

---

### 3. Loss

**`nn.CrossEntropyLoss(weight=None, reduction='mean', label_smoothing=0.0)`**  
- **What it is:** Combines log_softmax and NLL for multi-class classification. Input: logits (no softmax); target: class indices (long tensor).  
- **How used:** `criterion = nn.CrossEntropyLoss()` then `loss = criterion(logits, targets)`.  
- **Parameters:** `weight`: class weights; `reduction`: `'mean'`, `'sum'`, or `'none'`; `label_smoothing`: optional smoothing.

**`nn.BCELoss(reduction='mean')`**  
- **What it is:** Binary cross-entropy for probabilities. Input and target in $[0,1]$.  
- **How used:** For binary classification when the model outputs a probability (e.g. after sigmoid).  
- **Parameters:** `reduction`: `'mean'`, `'sum'`, or `'none'`.

---

### 4. Optimizer

**`torch.optim.SGD(params, lr=..., momentum=0, dampening=0, weight_decay=0, nesterov=False)`**  
- **What it is:** Stochastic gradient descent; optionally with momentum $v_t = \mu v_{t-1} + g_t$, then $\theta \leftarrow \theta - \eta v_t$.  
- **How used:** `optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)`.  
- **Parameters:** `params`: iterable of parameters (usually `model.parameters()`); `lr`: learning rate; `momentum`: momentum factor; `weight_decay`: L2 penalty.

**`torch.optim.Adam(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`**  
- **What it is:** Adam (adaptive learning rate).  
- **Parameters:** `params`, `lr`, `betas` (for momentum and scale), `eps` (numerical stability), `weight_decay`.

---

### 5. Training loop (per batch)

- **`optimizer.zero_grad()`** — Clears old gradients. Call once per batch so gradients are not accumulated from previous batches.  
- **`loss = criterion(model(x), y)`** — Forward pass and loss computation.  
- **`loss.backward()`** — Computes $\frac{\partial L}{\partial \theta}$ for all parameters; gradients are stored in `.grad`.  
- **`optimizer.step()`** — Updates parameters using the current gradients (e.g. $\theta \leftarrow \theta - \eta \nabla_\theta L$ for SGD).

---

### 6. Evaluation

- **`model.eval()`** — Sets the model to evaluation mode (e.g. disables dropout, fixes BatchNorm stats). No parameters.  
- **`with torch.no_grad():`** — Disables gradient computation for the block; use when computing validation/test loss or accuracy to save memory and speed.  
- Then run forward passes and compute accuracy/loss on the validation set without calling `backward()` or `step()`.
---
